In [1]:
import os
os.environ['HF_HOME'] = "/ssd_scratch/yuvraj.bodana/huggingface"
os.environ['HF_DATASETS_CACHE'] = "/ssd_scratch/yuvraj.bodana/huggingface"

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationC
import torch
import torch.nn as nn

ImportError: cannot import name 'GenerationC' from 'transformers' (/scratch/yuvraj.bodana/myenv/lib/python3.12/site-packages/transformers/__init__.py)

In [3]:
model_1_name = "meta-llama/Llama-3.2-3B-Instruct"
model_2_name = "meta-llama/Llama-3.2-1B-Instruct"

In [4]:
model_1 = AutoModelForCausalLM.from_pretrained(model_2_name, device_map="auto", dtype=torch.float16, attn_implementation="eager")
tokenizer = AutoTokenizer.from_pretrained(model_2_name)

In [5]:
model_2 = AutoModelForCausalLM.from_pretrained(model_2_name, device_map="auto", dtype=torch.float16, attn_implementation="eager")
tokenizer_2 = AutoTokenizer.from_pretrained(model_2_name)

In [6]:
tokenizer_2.pad_token = tokenizer.eos_token
tokenizer_2.padding_side = "left"

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

In [7]:
model_2

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [8]:
sentences_en = [
    "How are you?",
    "I am fine.",
    "What is your name?",
    "Good morning!",
    "See you later."
]

In [9]:
def make_prompts(text):
    prompt = f"""Translate the following sentence from English to Hindi:
{text}
RESPONSE FORMAT: <ONLY TRANSLATED TEXT>"""

    messages = [
        {"role":"user", "content":prompt}
    ]

    return messages

In [17]:
messages = []
for i in sentences_en:
    messages.append(make_prompts(i))

In [11]:
prompt = """Translate the following sentence from English to Hindi:
It was a nice day. I hope the same for tomorrow."""
messages = [
    {"role": "user", "content": prompt},
]

* Input for Model 1

In [18]:
inputs = tokenizer_2.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    padding=True
).to(model_2.device)

In [19]:
with torch.inference_mode():
    outputs = model_2.generate(**inputs, max_new_tokens=128, output_hidden_states=True, return_dict_in_generate=True)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [20]:
generated_ids = outputs.sequences

In [21]:
for index,i in enumerate(generated_ids):
    # print(inputs['input_ids'])
    print(len(i[inputs["input_ids"][index].shape[-1]:]))

39
39
39
39
39


In [31]:
generated_ids.detach().cpu().numpy()[0][26]

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [22]:
for index,i in enumerate(generated_ids):
    print(tokenizer_2.decode(i[inputs["input_ids"][index].shape[-1]:], skip_special_tokens=False))
    print("---")

मैं आपको सही हिंदी में मूल की जानकारी नहीं दे सकता<|eot_id|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|>
---
मैं ठीक हूँ<|eot_id|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|>
---
अपना नाम क्या है?<|eot_id|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_tex

In [23]:
import torch

# Assuming you already ran:
# outputs = model_2.generate(**inputs, max_new_tokens=128, output_hidden_states=True, return_dict_in_generate=True)

# 1. Isolate the hidden states of the generated tokens (skip the prompt at index 0)
generated_steps = outputs.hidden_states[1:] 

# 2. Select which layer you want to analyze (e.g., -1 for the final layer)
layer_idx = -1

# 3. Extract the specific layer's tensor for each generation step
# Each step in 'generated_steps' is a tuple of layers. 
# step[layer_idx] gives a tensor of shape: (batch_size, 1, hidden_dim)
layer_states_per_step = [step[layer_idx] for step in generated_steps]

# 4. Concatenate all steps along the sequence dimension (dim=1)
# Resulting shape: (batch_size, num_generated_tokens, hidden_dim)
stacked_output_states = torch.cat(layer_states_per_step, dim=1)

print(f"Shape of all generated hidden states: {stacked_output_states.shape}")

Shape of all generated hidden states: torch.Size([5, 38, 2048])


In [24]:
tokenizer.eos_token_id

128009

In [26]:
import torch

# Assuming outputs.sequences contains the full token IDs (prompt + generated)
generated_token_ids = outputs.sequences[:, inputs['input_ids'].shape[1]:]

# Force the pad_id to be the exact ID of the <|end_of_text|> token
pad_id = tokenizer.convert_tokens_to_ids("<|end_of_text|>")

# Now the sum will correctly ignore the padding tokens
sequence_lengths = (generated_token_ids != pad_id).sum(dim=1) - 1

print(sequence_lengths)
# This should now output varying indices like: tensor([ 5,  4,  5,  2,  5 ], device='cuda:0')

# Find the index of the last non-padding token for each sequence
# This assumes you have a defined pad_token_id
# pad_id = tokenizer.pad_token_id

# sequence_lengths = (generated_token_ids != pad_id).sum(dim=1) - 1

# Create an index tensor to pluck the correct hidden state for each batch item
batch_indices = torch.arange(stacked_output_states.size(0))

# Extract the true final token for each sequence
true_last_token_states = stacked_output_states[batch_indices, sequence_lengths, :]

# Shape is now (batch_size, hidden_dim) without any padding corruption
print(true_last_token_states.shape)

tensor([26,  6, 10,  4, 38], device='cuda:0')
torch.Size([5, 2048])


In [28]:
generation_hidden_states = outputs.hidden_states

In [80]:
generation_hidden_states[1][-1].shape

torch.Size([5, 1, 2048])

In [74]:
len(generation_hidden_states)

108

In [75]:
display(generation_hidden_states[-1][-1])
display(generation_hidden_states[-1][-1].shape)

tensor([[[ 0.2284,  2.2676,  1.2109,  ..., -4.1289, -4.0078, -1.7246]],

        [[ 1.2490,  5.3047, -0.2454,  ...,  4.2461, -2.2363, -3.3418]],

        [[ 1.2168,  6.5078,  0.2015,  ...,  3.9844, -1.3418, -1.2793]],

        [[ 0.6709,  4.1250, -0.4023,  ...,  4.4375, -1.7109, -3.6621]],

        [[ 1.3389,  6.3164, -0.2559,  ...,  4.2148, -1.5146, -2.3613]]],
       device='cuda:0', dtype=torch.float16)

torch.Size([5, 1, 2048])

### Pruning

In [16]:
original_layers = list(model_1.model.layers) 

layers_to_drop = [0] 

new_layers = [layer for i, layer in enumerate(original_layers) if i not in layers_to_drop]

model_1.model.layers = nn.ModuleList(new_layers)

model_1.config.num_hidden_layers = len(new_layers)

In [17]:
for index, layer in enumerate(model_1.model.layers):
            # Update the index in the attention module (where the error originates)
            if hasattr(layer, "self_attn"):
                layer.self_attn.layer_idx = index

            # Update the index on the layer itself (good practice for compatibility)
            if hasattr(layer, "layer_idx"):
                layer.layer_idx = index

In [23]:
model_1._temporary_cache = None
model_1.config.use_cache = False

In [18]:
model_1

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-14): 15 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [19]:
inputs_2 = tokenizer_2.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model_1.device)

In [20]:
with torch.inference_mode():
    outputs_2 = model_1.generate(**inputs_2, max_new_tokens=128, output_hidden_states=True, return_dict_in_generate=True)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [21]:
generated_ids_2 = outputs_2.sequences

In [22]:
len(generated_ids_2[0][inputs_2["input_ids"].shape[-1]:])

128

In [23]:
print(tokenizer_2.decode(generated_ids_2[0][inputs_2["input_ids"].shape[-1]:])) 

-z-m-z-E/.m-b-paper-d-z.r.r.e.org..org-z.m-z-z-c.-z.com.e-c.com.sz.spsssssssssssssssssssssssssssssssssssssssssssssssssssss2,222222221132.j1)
2.j22.2219325cs12e2132112122


In [24]:
generation_hidden_states_2 = outputs_2.hidden_states

In [25]:
len(generation_hidden_states_2)

128

In [37]:
generation_hidden_states_2[-1].shape

AttributeError: 'tuple' object has no attribute 'shape'

In [27]:
generation_hidden_states[-1][-1].shape

torch.Size([1, 1, 2048])

In [28]:
from cka_metrics import cka, gram_linear, feature_space_linear_cka

In [39]:
base_embed = generation_hidden_states[-1][-1][0].detach().cpu().numpy()
prune_embed = generation_hidden_states_2[-1][-1][0].detach().cpu().numpy()

In [30]:
base_embed.shape, prune_embed.shape

((1, 2048), (1, 2048))

In [36]:
gram_linear(base_embed), gram_linear(prune_embed)

(array([[10704.]], dtype=float16), array([[6992.]], dtype=float16))

In [31]:
cka_score = cka(gram_linear(base_embed), gram_linear(prune_embed))
print(f"CKA Score is {cka_score}")

The values are: 0.0 and 0.0
CKA Score is nan


/home2/yuvrajsinh.bodana/Pruning/EXP/cka_metrics.py:94: RuntimeWarning: invalid value encountered in scalar divide
  return scaled_hsic / (normalization_x * normalization_y)


In [32]:
feature_space_linear_cka(base_embed, prune_embed)

/home2/yuvrajsinh.bodana/Pruning/EXP/cka_metrics.py:148: RuntimeWarning: invalid value encountered in scalar divide
  return dot_product_similarity / (normalization_x * normalization_y)


nan

In [44]:
prompt_3b = generation_hidden_states[0][-1]
gen_3b = [generation_hidden_states[i][-1] for i in range(1, len(generation_hidden_states))]
full_3b_tensor = torch.cat([prompt_3b] + gen_3b, dim=1)

# 2. Reconstruct full sequence for 1B last layer
prompt_1b = generation_hidden_states_2[0][-1]
gen_1b = [generation_hidden_states_2[i][-1] for i in range(1, len(generation_hidden_states_2))]
full_1b_tensor = torch.cat([prompt_1b] + gen_1b, dim=1)

In [42]:
import numpy as np
def gram_linear(X):
    return X @ X.T
def center_gram(K):
    n = K.shape[0]
    H = np.eye(n) - np.ones((n, n)) / n
    return H @ K @ H
def linear_CKA(X, Y):
    Kx = center_gram(gram_linear(X))
    Ky = center_gram(gram_linear(Y))
    hsic = np.trace(Kx @ Ky)
    normalization = np.linalg.norm(Kx, 'fro') * np.linalg.norm(Ky, 'fro')
    return hsic / normalization if normalization != 0 else float("nan")
# # Compute CKA score
# cka_score = linear_CKA(embeddings_en, embeddings_fr)
# # Output
# print(f"Average Cosine Similarity: {avg_cosine_similarity:.4f}")
# print(f"CKA Score: {cka_score:.4f}")


In [50]:
import numpy as np

def gram_linear(x):
  """Compute Gram (kernel) matrix for a linear kernel.

  Args:
    x: A num_examples x num_features matrix of features.

  Returns:
    A num_examples x num_examples Gram matrix of examples.
  """
  return x.dot(x.T)


def gram_rbf(x, threshold=1.0):
  """Compute Gram (kernel) matrix for an RBF kernel.

  Args:
    x: A num_examples x num_features matrix of features.
    threshold: Fraction of median Euclidean distance to use as RBF kernel
      bandwidth. (This is the heuristic we use in the paper. There are other
      possible ways to set the bandwidth; we didn't try them.)

  Returns:
    A num_examples x num_examples Gram matrix of examples.
  """
  dot_products = x.dot(x.T)
  sq_norms = np.diag(dot_products)
  sq_distances = -2 * dot_products + sq_norms[:, None] + sq_norms[None, :]
  sq_median_distance = np.median(sq_distances)
  return np.exp(-sq_distances / (2 * threshold ** 2 * sq_median_distance))


def center_gram(gram, unbiased=False):
  """Center a symmetric Gram matrix.

  This is equvialent to centering the (possibly infinite-dimensional) features
  induced by the kernel before computing the Gram matrix.

  Args:
    gram: A num_examples x num_examples symmetric matrix.
    unbiased: Whether to adjust the Gram matrix in order to compute an unbiased
      estimate of HSIC. Note that this estimator may be negative.

  Returns:
    A symmetric matrix with centered columns and rows.
  """
  if not np.allclose(gram, gram.T):
    raise ValueError('Input must be a symmetric matrix.')
  gram = gram.copy()

  if unbiased:
    # This formulation of the U-statistic, from Szekely, G. J., & Rizzo, M.
    # L. (2014). Partial distance correlation with methods for dissimilarities.
    # The Annals of Statistics, 42(6), 2382-2412, seems to be more numerically
    # stable than the alternative from Song et al. (2007).
    n = gram.shape[0]
    np.fill_diagonal(gram, 0)
    means = np.sum(gram, 0, dtype=np.float64) / (n - 2)
    means -= np.sum(means) / (2 * (n - 1))
    gram -= means[:, None]
    gram -= means[None, :]
    np.fill_diagonal(gram, 0)
  else:
    means = np.mean(gram, 0, dtype=np.float64)
    means -= np.mean(means) / 2
    gram -= means[:, None]
    gram -= means[None, :]

  return gram


def cka(gram_x, gram_y, debiased=False):
  """Compute CKA.

  Args:
    gram_x: A num_examples x num_examples Gram matrix.
    gram_y: A num_examples x num_examples Gram matrix.
    debiased: Use unbiased estimator of HSIC. CKA may still be biased.

  Returns:
    The value of CKA between X and Y.
  """
  gram_x = center_gram(gram_x, unbiased=debiased)
  gram_y = center_gram(gram_y, unbiased=debiased)

  # Note: To obtain HSIC, this should be divided by (n-1)**2 (biased variant) or
  # n*(n-3) (unbiased variant), but this cancels for CKA.
  scaled_hsic = gram_x.ravel().dot(gram_y.ravel())

  normalization_x = np.linalg.norm(gram_x)
  normalization_y = np.linalg.norm(gram_y)
  return scaled_hsic / (normalization_x * normalization_y)


def _debiased_dot_product_similarity_helper(
    xty, sum_squared_rows_x, sum_squared_rows_y, squared_norm_x, squared_norm_y,
    n):
  """Helper for computing debiased dot product similarity (i.e. linear HSIC)."""
  # This formula can be derived by manipulating the unbiased estimator from
  # Song et al. (2007).
  return (
      xty - n / (n - 2.) * sum_squared_rows_x.dot(sum_squared_rows_y)
      + squared_norm_x * squared_norm_y / ((n - 1) * (n - 2)))


def feature_space_linear_cka(features_x, features_y, debiased=False):
  """Compute CKA with a linear kernel, in feature space.

  This is typically faster than computing the Gram matrix when there are fewer
  features than examples.

  Args:
    features_x: A num_examples x num_features matrix of features.
    features_y: A num_examples x num_features matrix of features.
    debiased: Use unbiased estimator of dot product similarity. CKA may still be
      biased. Note that this estimator may be negative.

  Returns:
    The value of CKA between X and Y.
  """
  features_x = features_x - np.mean(features_x, 0, keepdims=True)
  features_y = features_y - np.mean(features_y, 0, keepdims=True)

  dot_product_similarity = np.linalg.norm(features_x.T.dot(features_y)) ** 2
  normalization_x = np.linalg.norm(features_x.T.dot(features_x))
  normalization_y = np.linalg.norm(features_y.T.dot(features_y))

  if debiased:
    n = features_x.shape[0]
    # Equivalent to np.sum(features_x ** 2, 1) but avoids an intermediate array.
    sum_squared_rows_x = np.einsum('ij,ij->i', features_x, features_x)
    sum_squared_rows_y = np.einsum('ij,ij->i', features_y, features_y)
    squared_norm_x = np.sum(sum_squared_rows_x)
    squared_norm_y = np.sum(sum_squared_rows_y)

    dot_product_similarity = _debiased_dot_product_similarity_helper(
        dot_product_similarity, sum_squared_rows_x, sum_squared_rows_y,
        squared_norm_x, squared_norm_y, n)
    normalization_x = np.sqrt(_debiased_dot_product_similarity_helper(
        normalization_x ** 2, sum_squared_rows_x, sum_squared_rows_x,
        squared_norm_x, squared_norm_x, n))
    normalization_y = np.sqrt(_debiased_dot_product_similarity_helper(
        normalization_y ** 2, sum_squared_rows_y, sum_squared_rows_y,
        squared_norm_y, squared_norm_y, n))

  return dot_product_similarity / (normalization_x * normalization_y)

In [49]:
cka_score = linear_CKA(generation_hidden_states[0][-1], generation_hidden_states_2[0][-1])

RuntimeError: Expected size for first two dimensions of batch2 tensor to be: [2048, 2048] but got: [2048, 57].

In [58]:
cka(gram_linear(generation_hidden_states[-1][-1].detach().cpu().numpy()), gram_linear(generation_hidden_states_2[-1][-1].detach().cpu().numpy()))

ValueError: shapes (1,1,2048) and (2048,1,1) not aligned: 2048 (dim 2) != 1 (dim 1)

In [59]:
feature_space_linear_cka(generation_hidden_states[-1][-1].detach().cpu().numpy(), generation_hidden_states_2[-1][-1].detach().cpu().numpy())

/tmp/ipykernel_151280/2487560576.py:147: RuntimeWarning: invalid value encountered in scalar divide
  return dot_product_similarity / (normalization_x * normalization_y)


nan

In [60]:
import torch

# 1. Gather the final layer output across ALL generation steps (all tokens)
# Loop through the generation steps and pull the very last layer [-1] from each step
states_base = [generation_hidden_states[i][-1] for i in range(len(generation_hidden_states))]
states_dropped = [generation_hidden_states_2[i][-1] for i in range(len(generation_hidden_states_2))]

# 2. Determine the minimum matching length to prevent shape mismatches
min_len = min(len(states_base), len(states_dropped))

# 3. Stack into a continuous 3D tensor and squeeze to a 2D matrix (num_tokens, 2048)
# We slice [:min_len] so they have the exact same number of samples
tensor_x = torch.cat(states_base[:min_len], dim=1).squeeze(0)
tensor_y = torch.cat(states_dropped[:min_len], dim=1).squeeze(0)

# 4. Convert safely to NumPy
features_x = tensor_x.detach().cpu().to(torch.float32).numpy()
features_y = tensor_y.detach().cpu().to(torch.float32).numpy()

# 5. Run feature space CKA (Will output a valid score because samples > 1)
cka_score = feature_space_linear_cka(features_x, features_y)
print("Valid CKA Score across all generated tokens:", cka_score)


Valid CKA Score across all generated tokens: 0.5070686815099136


In [61]:
len(states_base), len(states_dropped)

(38, 128)

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("./pruned_model_cos_rec.tsv", sep='\t')

In [4]:
for index, row in df.iterrows():
    print(row.Source)
    print("---")
    print(row.Reference)
    print("---")
    print(row.MT)
    print("= = "*20)

All animals that originally arrived in the islands came here either by swimming, flying or floating.
---
द्वीपों में मूल रूप से आए सभी जानवर यहाँ तैरकर, उड़कर या बहकर आए हैं.
---
मात्र जिन्होंने बीचले बीचले परिवारों को पहुंचे थे उनका पहला वहाँ से पहुंचा था जो शीघ्र से पहुंच कर के हो सकते थे।
= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 
Due to the long distance from the continent mammals were unable to make the journey making the giant tortoise the primary grazing animal in the Galapagos.
---
महाद्वीप से लंबी दूरी होने के कारण, स्तनधारी यात्रा करने में असमर्थ थे जिसने विशाल कछुए को गैलापागोस में चरने वाला प्राथमिक जानवर बना दिया।
---
गालापागोस के पशुपालकों की दूरी से कोई भी स्थान नहीं होता था क्योंकि यह कुत्ते द्वारा घोड़े का सबसे बड़ा मांगी पशुपालक था जो गालापागोस के पशुपालकों का मुख्य खाने वाला था।
= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 
Since the arrival of man to the Galapagos, many mammals have been introd

In [8]:
import evaluate
sacrebleu = evaluate.load("sacrebleu")

In [9]:
results = sacrebleu.compute(predictions=df['MT'], references=df['Reference'])

In [10]:
results

{'score': 1.2757586277016335,
 'counts': [61, 8, 1, 0],
 'totals': [325, 315, 305, 295],
 'precisions': [18.76923076923077,
  2.5396825396825395,
  0.32786885245901637,
  0.1694915254237288],
 'bp': 1.0,
 'sys_len': 325,
 'ref_len': 240}

In [14]:
df = pd.read_csv("./pruned_model_cka_rec.tsv", sep='\t')

In [15]:
for index, row in df.iterrows():
    print(row.Source)
    print("---")
    print(row.Reference)
    print("---")
    print(row.MT)
    print("= = "*20)

All animals that originally arrived in the islands came here either by swimming, flying or floating.
---
द्वीपों में मूल रूप से आए सभी जानवर यहाँ तैरकर, उड़कर या बहकर आए हैं.
---
इन द्वीपों में आने वाले सभी जाने-माने जाने वाले जाने वाले जाने वाले जाने वाले जाने वाले जाने वाले जाने वाले जाने वाले जाने वाले जाने वाले जाने वाले जाने वाले जाने वाले जाने वाले जाने वाले जाने वाले जाने वाले जाने वाले
= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 
Due to the long distance from the continent mammals were unable to make the journey making the giant tortoise the primary grazing animal in the Galapagos.
---
महाद्वीप से लंबी दूरी होने के कारण, स्तनधारी यात्रा करने में असमर्थ थे जिसने विशाल कछुए को गैलापागोस में चरने वाला प्राथमिक जानवर बना दिया।
---
बिल्ली के देश से एक लंबी दूरी से मानवों का पालन करने में कठिनता हुई कि वह गालापागोस के क्षेत्र में एक बड़े घाते घाती था जिसके कारण मोटे घाती था।
= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 

In [16]:
results = sacrebleu.compute(predictions=df['MT'], references=df['Reference'])
results

{'score': 2.489110082767928,
 'counts': [72, 18, 3, 0],
 'totals': [282, 272, 262, 252],
 'precisions': [25.53191489361702,
  6.617647058823529,
  1.1450381679389312,
  0.1984126984126984],
 'bp': 1.0,
 'sys_len': 282,
 'ref_len': 240}

In [17]:
df = pd.read_csv("./pruned_model_cka_rec_opt.tsv", sep='\t')

for index, row in df.iterrows():
    print(row.Source)
    print("---")
    print(row.Reference)
    print("---")
    print(row.MT)
    print("= = "*20)

results = sacrebleu.compute(predictions=df['MT'], references=df['Reference'])
results

All animals that originally arrived in the islands came here either by swimming, flying or floating.
---
द्वीपों में मूल रूप से आए सभी जानवर यहाँ तैरकर, उड़कर या बहकर आए हैं.
---
अभि राज्यों में प्रारूपांतरण करने के लिए लोगों का प्रवास आया हुआ है और शीतके माहमान हैं।
= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 
Due to the long distance from the continent mammals were unable to make the journey making the giant tortoise the primary grazing animal in the Galapagos.
---
महाद्वीप से लंबी दूरी होने के कारण, स्तनधारी यात्रा करने में असमर्थ थे जिसने विशाल कछुए को गैलापागोस में चरने वाला प्राथमिक जानवर बना दिया।
---
मुन्ना जानी जानी पर जानी गिरा जानी के लिए कि कि कि गालोपास की लंबी दूरी के कारण कि गिरा जानी पर विशिष्ट कारण कि गिरा जानी के लिए कि कि देशों की पर्याक्ष्य में चिक्के की जानी पर कि देशों की पर्याक्ष्य में कि देशों का पारित करने के ल
= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 
Since the arrival of man to the Gal

{'score': 0.9716062536939682,
 'counts': [45, 4, 0, 0],
 'totals': [225, 215, 205, 195],
 'precisions': [20.0,
  1.8604651162790697,
  0.24390243902439024,
  0.1282051282051282],
 'bp': 0.9355069850316178,
 'sys_len': 225,
 'ref_len': 240}